# CQD Bronze -> Silver -> Gold

Reads immutable call-record snapshots from the ADLS Bronze container and builds Delta tables in the attached Lakehouse for Direct Lake.

- **Silver:** `silver_calls`, `silver_participants`, `silver_sessions`, `silver_segments`, `silver_mediastreams`. Latest Graph version per call only.
- **Gold:** `gold_stream_quality`, `gold_call_summary`, rebuilt from Silver.
- **Privacy:** users appear only as salted SHA-256 keys. Names, UPNs, and IP addresses are not written. Adding identified attributes needs explicit approval.
- **Idempotent:** child tables are replaced per call before `silver_calls` is merged last, so a rerun after a failure repairs the data. Older or equal versions are skipped.

Attach a Lakehouse as the default before running. Quality thresholds are illustrative, not verified CQD reporting equivalence.

## Parameters

In [ ]:
# Bronze container path, for example abfss://bronze@<account>.dfs.core.usgovcloudapi.net
BRONZE_ROOT = ""
TENANT_ID = ""
# Only Bronze files modified within this window are processed
LOOKBACK_HOURS = 48
# Set from an approved secret store for a stable, non-reversible user key; do not hard-code
USER_HASH_SALT = ""

## Parsing helpers

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone

DURATION = re.compile(r"^P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+(?:\.\d+)?)S)?)?$")


def millis(value):
    match = DURATION.match(value or "")
    if not match:
        return None
    days, hours, minutes, secs = match.groups()
    total = int(days or 0) * 86400 + int(hours or 0) * 3600 + int(minutes or 0) * 60 + float(secs or 0)
    return round(total * 1000, 3)


def as_float(value):
    return None if value is None else float(value)


def parse_ts(text):
    if not text:
        return None
    text = re.sub(r"(\.\d{6})\d+", r"\1", text.replace("Z", "+00:00"))
    parsed = datetime.fromisoformat(text)
    return parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)


def seconds_between(start, end):
    if start is None or end is None:
        return None
    return round((end - start).total_seconds(), 3)


def user_key(identity_id):
    if not identity_id:
        return None
    return "U-" + hashlib.sha256((USER_HASH_SALT + identity_id).encode()).hexdigest()[:16]


def identity_id(identity):
    identity = identity or {}
    for node in (identity, identity.get("user") or {}, identity.get("guest") or {}):
        if node.get("id"):
            return node["id"]
    return None


def endpoint(ep):
    ep = ep or {}
    agent = ep.get("userAgent") or {}
    return (
        user_key(identity_id(ep.get("associatedIdentity") or ep.get("identity"))),
        agent.get("platform"),
        agent.get("productFamily"),
    )


def flatten_snapshot(snap):
    """Return one tuple list per table; every row carries (CallId, CallVersion) first."""
    r = snap["record"]
    cid, ver = r["id"], r.get("version")
    out = {"calls": [], "participants": [], "sessions": [], "segments": [], "mediastreams": []}
    start, end = parse_ts(r.get("startDateTime")), parse_ts(r.get("endDateTime"))
    organizer = user_key(identity_id((r.get("organizer_v2") or {}).get("identity") or r.get("organizer")))
    out["calls"].append((
        cid, ver, r.get("type"), ",".join(r.get("modalities") or []), start, end,
        seconds_between(start, end), organizer, parse_ts(snap.get("retrieved_at")),
    ))
    for p in r.get("participants_v2") or []:
        ident = p.get("identity") or {}
        pid = identity_id(ident)
        key = user_key(pid or p.get("id"))
        out["participants"].append((cid, ver, key, bool(ident.get("guest")), bool(pid) and user_key(pid) == organizer))
    for s in r.get("sessions") or []:
        sid = s["id"]
        s_start, s_end = parse_ts(s.get("startDateTime")), parse_ts(s.get("endDateTime"))
        caller_key, caller_platform, caller_product = endpoint(s.get("caller"))
        _, callee_platform, callee_product = endpoint(s.get("callee"))
        fail = s.get("failureInfo") or {}
        out["sessions"].append((
            cid, ver, sid, ",".join(s.get("modalities") or []), s_start, s_end,
            seconds_between(s_start, s_end), caller_key, caller_platform, caller_product,
            callee_platform, callee_product, fail.get("stage"), fail.get("reason"),
        ))
        for seg in s.get("segments") or []:
            gid = seg["id"]
            g_start, g_end = parse_ts(seg.get("startDateTime")), parse_ts(seg.get("endDateTime"))
            sfail = seg.get("failureInfo") or {}
            out["segments"].append((
                cid, ver, sid, gid, g_start, g_end, seconds_between(g_start, g_end),
                sfail.get("stage"), sfail.get("reason"),
            ))
            for media in seg.get("media") or []:
                cn, en = media.get("callerNetwork") or {}, media.get("calleeNetwork") or {}
                for st in media.get("streams") or []:
                    out["mediastreams"].append((
                        cid, ver, sid, gid, st.get("streamId"), media.get("label"), st.get("streamDirection"),
                        st.get("audioCodec"), st.get("videoCodec"),
                        millis(st.get("averageJitter")), millis(st.get("maxJitter")),
                        millis(st.get("averageRoundTripTime")), millis(st.get("maxRoundTripTime")),
                        as_float(st.get("averagePacketLossRate")), as_float(st.get("maxPacketLossRate")),
                        as_float(st.get("averageRatioOfConcealedSamples")), as_float(st.get("maxRatioOfConcealedSamples")),
                        as_float(st.get("averageAudioDegradation")), st.get("packetUtilization"),
                        cn.get("connectionType"), cn.get("wifiBand"), en.get("connectionType"), en.get("wifiBand"),
                        cn.get("networkTransportProtocol") or en.get("networkTransportProtocol"),
                    ))
    return out

## Schemas

In [ ]:
from pyspark.sql.types import (
    BooleanType, DoubleType, LongType, StringType, StructField, StructType, TimestampType,
)


def schema(*fields):
    return StructType([StructField(name, kind(), True) for name, kind in fields])


SCHEMAS = {
    "calls": schema(
        ("CallId", StringType), ("CallVersion", LongType), ("CallType", StringType), ("Modalities", StringType),
        ("StartDateTime", TimestampType), ("EndDateTime", TimestampType), ("DurationSeconds", DoubleType),
        ("OrganizerKey", StringType), ("RetrievedAt", TimestampType),
    ),
    "participants": schema(
        ("CallId", StringType), ("CallVersion", LongType), ("ParticipantKey", StringType),
        ("IsGuest", BooleanType), ("IsOrganizer", BooleanType),
    ),
    "sessions": schema(
        ("CallId", StringType), ("CallVersion", LongType), ("SessionId", StringType), ("Modalities", StringType),
        ("StartDateTime", TimestampType), ("EndDateTime", TimestampType), ("DurationSeconds", DoubleType),
        ("CallerKey", StringType), ("CallerPlatform", StringType), ("CallerProduct", StringType),
        ("CalleePlatform", StringType), ("CalleeProduct", StringType),
        ("FailureStage", StringType), ("FailureReason", StringType),
    ),
    "segments": schema(
        ("CallId", StringType), ("CallVersion", LongType), ("SessionId", StringType), ("SegmentId", StringType),
        ("StartDateTime", TimestampType), ("EndDateTime", TimestampType), ("DurationSeconds", DoubleType),
        ("FailureStage", StringType), ("FailureReason", StringType),
    ),
    "mediastreams": schema(
        ("CallId", StringType), ("CallVersion", LongType), ("SessionId", StringType), ("SegmentId", StringType),
        ("StreamId", StringType), ("MediaLabel", StringType), ("Direction", StringType),
        ("AudioCodec", StringType), ("VideoCodec", StringType),
        ("AvgJitterMs", DoubleType), ("MaxJitterMs", DoubleType),
        ("AvgRoundTripMs", DoubleType), ("MaxRoundTripMs", DoubleType),
        ("AvgPacketLossRate", DoubleType), ("MaxPacketLossRate", DoubleType),
        ("AvgConcealedRatio", DoubleType), ("MaxConcealedRatio", DoubleType),
        ("AvgAudioDegradation", DoubleType), ("PacketUtilization", LongType),
        ("CallerConnectionType", StringType), ("CallerWifiBand", StringType),
        ("CalleeConnectionType", StringType), ("CalleeWifiBand", StringType),
        ("TransportProtocol", StringType),
    ),
}

## Read Bronze and parse

In [ ]:
from datetime import timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window

if not BRONZE_ROOT or not TENANT_ID:
    raise ValueError("Set BRONZE_ROOT and TENANT_ID")

modified_after = (datetime.now(timezone.utc) - timedelta(hours=LOOKBACK_HOURS)).strftime("%Y-%m-%dT%H:%M:%S")
source = f"{BRONZE_ROOT.rstrip('/')}/callrecords/{TENANT_ID}"

files = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.json")
    .option("recursiveFileLookup", "true")
    .option("modifiedAfter", modified_after)
    .load(source)
    .select("content")
)

parsed = files.rdd.map(lambda row: flatten_snapshot(json.loads(bytes(row.content)))).persist()


def table_df(name):
    return spark.createDataFrame(parsed.flatMap(lambda out: out[name]), SCHEMAS[name])


# Keep only the latest version of each call within this batch
calls = table_df("calls")
latest = Window.partitionBy("CallId").orderBy(F.col("CallVersion").desc(), F.col("RetrievedAt").desc())
calls = calls.withColumn("rn", F.row_number().over(latest)).filter("rn = 1").drop("rn")

## Skip versions already in Silver

In [ ]:
SILVER = {name: f"silver_{name}" for name in SCHEMAS}


def delta_ready(name):
    """True only if the table is registered and its Delta data exists; drops a broken registration."""
    if not spark.catalog.tableExists(name):
        return False
    try:
        from delta.tables import DeltaTable

        DeltaTable.forName(spark, name)
        return True
    except Exception as exc:
        if "DELTA_PATH_DOES_NOT_EXIST" in str(exc) or "doesn't exist" in str(exc) or "is not a Delta table" in str(exc):
            print(name, "is registered but has no Delta data; recreating")
            spark.sql(f"DROP TABLE IF EXISTS {name}")
            return False
        raise


if delta_ready(SILVER["calls"]):
    existing = spark.table(SILVER["calls"]).select(
        "CallId", F.col("CallVersion").alias("ExistingVersion")
    )
    calls = (
        calls.join(existing, "CallId", "left")
        .filter("ExistingVersion IS NULL OR CallVersion > ExistingVersion")
        .drop("ExistingVersion")
    )

calls = calls.persist()
batch_keys = calls.select("CallId", "CallVersion")
print("Calls to load:", batch_keys.count())

## Write Silver (children first, `silver_calls` last)

In [ ]:
from delta.tables import DeltaTable


def write_silver(target, df, update_existing):
    try:
        if delta_ready(target):
            update_existing()
        else:
            df.write.format("delta").mode("overwrite").saveAsTable(target)
    except Exception as exc:
        if "DELTA_PATH_DOES_NOT_EXIST" not in str(exc):
            raise
        # Registered in the catalog without Delta data (for example after an interrupted first write)
        print(target, "has no Delta data; recreating")
        spark.sql(f"DROP TABLE IF EXISTS {target}")
        df.write.format("delta").mode("overwrite").saveAsTable(target)
    print(target, "updated")


def replace_children(name):
    df = table_df(name).join(batch_keys, ["CallId", "CallVersion"], "inner")
    target = SILVER[name]

    def update_existing():
        ids = batch_keys.select("CallId")
        (
            DeltaTable.forName(spark, target).alias("t")
            .merge(ids.alias("s"), "t.CallId = s.CallId")
            .whenMatchedDelete()
            .execute()
        )
        df.write.format("delta").mode("append").saveAsTable(target)

    write_silver(target, df, update_existing)


for child in ("participants", "sessions", "segments", "mediastreams"):
    replace_children(child)


def upsert_calls():
    (
        DeltaTable.forName(spark, SILVER["calls"]).alias("t")
        .merge(calls.alias("s"), "t.CallId = s.CallId")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


write_silver(SILVER["calls"], calls, upsert_calls)

parsed.unpersist()
calls.unpersist()

## Gold (full rebuild from Silver)

Direct Lake reads tables, not views, so Gold is materialized. Quality thresholds: Poor = loss > 10%, RTT > 500 ms, jitter > 30 ms, or audio degradation > 1 (aligned with Microsoft CQD guidance as understood; verify). Fair = loss > 5%, RTT > 300 ms, or jitter > 15 ms (illustrative).

In [ ]:
spark.sql(
    """
    CREATE OR REPLACE TABLE gold_stream_quality USING DELTA AS
    SELECT
        m.*,
        s.CallerPlatform,
        c.StartDateTime AS CallStartDateTime,
        CASE
            WHEN m.AvgJitterMs IS NULL AND m.AvgRoundTripMs IS NULL AND m.AvgPacketLossRate IS NULL THEN 'No data'
            WHEN m.AvgPacketLossRate > 0.1 OR m.AvgRoundTripMs > 500 OR m.AvgJitterMs > 30 OR m.AvgAudioDegradation > 1 THEN 'Poor'
            WHEN m.AvgPacketLossRate > 0.05 OR m.AvgRoundTripMs > 300 OR m.AvgJitterMs > 15 THEN 'Fair'
            ELSE 'Good'
        END AS StreamQuality
    FROM silver_mediastreams m
    JOIN silver_sessions s ON s.SessionId = m.SessionId AND s.CallId = m.CallId
    JOIN silver_calls c ON c.CallId = m.CallId
    """
)

spark.sql(
    """
    CREATE OR REPLACE TABLE gold_call_summary USING DELTA AS
    SELECT
        c.CallId,
        c.CallVersion,
        c.CallType,
        c.Modalities,
        c.StartDateTime,
        DATE_FORMAT(c.StartDateTime, 'MMM d HH:mm:ss') AS CallLabel,
        c.DurationSeconds,
        COALESCE(sc.SessionCount, 0) AS SessionCount,
        COALESCE(pc.ParticipantCount, 0) AS ParticipantCount,
        AVG(g.AvgJitterMs) AS AvgJitterMs,
        AVG(g.AvgRoundTripMs) AS AvgRoundTripMs,
        AVG(g.AvgPacketLossRate) AS AvgPacketLossRate,
        MAX(g.MaxPacketLossRate) AS MaxPacketLossRate,
        COUNT_IF(g.StreamQuality = 'Good') AS GoodStreams,
        COUNT_IF(g.StreamQuality IN ('Fair', 'Poor')) AS FairOrPoorStreams
    FROM silver_calls c
    LEFT JOIN gold_stream_quality g ON g.CallId = c.CallId
    LEFT JOIN (SELECT CallId, COUNT(*) AS SessionCount FROM silver_sessions GROUP BY CallId) sc ON sc.CallId = c.CallId
    LEFT JOIN (SELECT CallId, COUNT(DISTINCT ParticipantKey) AS ParticipantCount FROM silver_participants GROUP BY CallId) pc ON pc.CallId = c.CallId
    GROUP BY c.CallId, c.CallVersion, c.CallType, c.Modalities, c.StartDateTime, c.DurationSeconds, sc.SessionCount, pc.ParticipantCount
    """
)

for table in list(SILVER.values()) + ["gold_stream_quality", "gold_call_summary"]:
    print(table, spark.table(table).count())